#### AI모델이 대화를 처리할 때, 처리해야 할 정보의 양을 줄여서 더 빠르고 효율적으로 응답할 수 있도록 하는 방법

In [1]:
import sys

# sys.path.append("C:/cloud_backend/langchain_app")
sys.path.append("D:/source/ai/langchain_app")

from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import trim_messages
from langchain_openai import ChatOpenAI
from memory_history import get_session_history
from operator import itemgetter
from dotenv import load_dotenv


load_dotenv("../../.env")

model = ChatOpenAI(model="gpt-4o-mini")



prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)


# 메시지 트리밍 유틸리티 설정( 가장 최근 2개의 메세지만 유지 )
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    #RunnablePassthrough.assign()은 반드시 딕셔너리(dict) 입력만 받음
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | model
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    # input_messages_key, history_messages_key 항목을 제공하지 않으면 메시지 리스트(list) 형태로 변환하여 반환됨
)

# 새로운 대화 내용 추가 후 체인 실행
answer = chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}  # 세션 ID 설정
).content
print(answer)


d:\source\ai\langchain_app\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


5년 내에 집을 구매하기 위한 재정 계획을 세우는 것은 중요한 결정입니다. 다음 단계들을 고려해보세요:

### 1. 목표 설정
- **예산 결정:** 예상하는 집 가격과 구매 지역을 정하세요. 그에 따라 필요한 자금을 계산합니다.
- **저축 목표:** 전체 집 가격에서 얼마나 많은 금액을 초기 계약금으로 준비해야 하는지 파악합니다. 일반적으로 계약금은 집 가격의 10%에서 20% 사이입니다.

### 2. 재정 점검
- **소득 및 지출 분석:** 현재의 소득과 지출을 파악하고, 얼마나 저축할 수 있는지를 계산합니다.
- **신용 점검:** 좋은 신용 점수는 대출 시 이자율에 많은 영향을 미칩니다. 신용 점수를 확인하고, 필요시 개선을 위한 조치를 취하세요.

### 3. 저축 계획
- **저축 계좌 개설:** 별도의 저축 계좌를 열어 집 구매 자금을 모아 두세요. 이자를 최대한 받을 수 있는 계좌를 선택하는 것이 좋습니다.
- **자동 저축 설정:** 매달 일정 금액을 자동으로 저축 계좌로 이체하는 방식으로 저축 습관을 들이는 것이 유리합니다.

### 4. 투자 고려
- **투자 전략:** 만약 집 구매 자금이 5년 이상 남아 있다면, 저축 뿐만 아니라 저위험 자산(예: 채권)에 투자해 자산을 늘리는 방법도 고려할 수 있습니다.

### 5. 예비 비용 고려
- **추가 비용:** 계약금 외에도 세금, 보험, 관리비와 같은 추가 비용이 발생합니다. 이러한 비용을 예산에 포함시켜야 합니다.

### 6. 대출 옵션 조사
- **대출 사전 승인:** 어떤 대출 상품이 적합한지 미리 알아보세요. 각 금융기관에서 제공하는 조건을 비교하고, 사전 승인을 받으면 유리합니다.

### 7. 전문가 상담
- **재정 상담:** 필요시 재정 전문가와 상담하여 보다 구체적이고 개인화된 전략을 마련하는 것도 좋은 방법입니다.

### 8. 계획 수정
- **정기적인 점검:** 매년 또는 매분기마다 재정 계획을 점검하고, 필요시 조정하세요. 상황에 따라 계획을 유연하게 수정

In [2]:
answer = chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}
).content
print(answer)

당신은 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하는지에 대한 질문을 하셨습니다. 제가 그에 대한 조언을 드렸습니다. 필요하신 점이나 더 궁금한 사항이 있으시면 말씀해 주세요!


In [3]:
answer = chain_with_trimmed_history.invoke(
    {"input": "요즘 유명한 K-POP 가수가 누구야?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}
).content
print(answer)

제 데이터는 2023년 10월까지의 정보로 업데이트되어 있으며, 그 이전까지의 K-POP 가수들에 대한 정보를 드릴 수 있습니다. 2023년에는 방탄소년단(BTS), 블랙핑크(Blackpink), 트와이스(Twice), 세븐틴(Seventeen) 등 여러 K-POP 그룹이 큰 인기를 끌었습니다. 이들은 전 세계적으로 많은 팬을 보유하고 있습니다. 최신 정보나 인기를 얻고 있는 신인 가수에 대해서는 최신 뉴스나 차트를 참고하시길 권장합니다.


In [4]:
answer = chain_with_trimmed_history.invoke(
    {"input": "내가 제일 처음 질문한 내용이 뭐지?"},  # 사용자의 질문
    {"configurable": {"session_id": "user01"}}
).content
print(answer)

당신이 가장 처음 한 질문은 "요즘 유명한 K-POP 가수가 누구야?"입니다.
